# 03. CoT Knowledge Distillation

교사 모델(GPT-4 계열)이 생성한 Chain-of-Thought 레이블을  
Self-Consistency 기반 품질 필터링으로 정제하는 과정입니다.  
스크립트: `src/02_cot_label_generation.py` → `src/03_cot_filtering.py`

In [1]:
print('=== CoT Label Generation Configuration ===')
print('Input          : golden_11900_stratified_samples.csv  (11,900 samples)')
print('Teacher model  : gpt-4o-mini')
print('Parallel workers: 5')
print('Max retries    : 3')
print('Output format  : JSONL  {uuid, input, scriptITN, output(CoT), status}')
print()
print('3-Step CoT reasoning format enforced:')
print('  1. 분석  : identify numeric/pattern expressions')
print('  2. 변환  : apply conversion rules step-by-step')
print('  3. 최종 문장 : {final ITN result}')

=== CoT Label Generation Configuration ===
Input          : golden_11900_stratified_samples.csv  (11,900 samples)
Teacher model  : gpt-4o-mini
Parallel workers: 5
Max retries    : 3
Output format  : JSONL  {uuid, input, scriptITN, output(CoT), status}

3-Step CoT reasoning format enforced:
  1. 분석  : identify numeric/pattern expressions
  2. 변환  : apply conversion rules step-by-step
  3. 최종 문장 : {final ITN result}


In [2]:
print('=== Example CoT Label (single sample) ===')
print()
print('[Input (scriptTN)]')
print('  조사결과에스에이치공사아파트의연도별평균분양가가이점이배올랐다고십오일에보도됐다.')
print()
print('[Ground Truth (scriptITN)]')
print('  조사결과SH공사아파트의연도별평균분양가가2.2배올랐다고15일에보도됐다.')
print()
print('[Teacher CoT Output]')
print('  ### 1. 분석:')
print("  입력 문장에서 숫자/패턴 표현으로 '에스에이치'(SH 발음), '이점이배'(2.2배),")
print("  '십오일'(15일)을 식별하였다. '에스에이치'는 영문 약자 발화이며,")
print("  '이점이'는 소수점 포함 수치, '십오'는 15를 의미하는 한국어 수사이다.")
print()
print('  ### 2. 변환:')
print("  '에스에이치' → 규칙 6(제품/혼합 패턴)에 따라 'SH'로 변환.")
print("  '이점이배' → 규칙 4(소수점 수치): '이점이' = 2.2, 결합하여 '2.2배'.")
print("  '십오일' → 규칙 1(날짜/시간): '십오' = 15, '일' = 일(日), '15일'.")
print()
print('  ### 3. 최종 문장:')
print('  {조사결과SH공사아파트의연도별평균분양가가2.2배올랐다고15일에보도됐다.}')
print()
print('[Status] OK  ← final sentence matches ground truth exactly')

=== Example CoT Label (single sample) ===

[Input (scriptTN)]
  조사결과에스에이치공사아파트의연도별평균분양가가이점이배올랐다고십오일에보도됐다.

[Ground Truth (scriptITN)]
  조사결과SH공사아파트의연도별평균분양가가2.2배올랐다고15일에보도됐다.

[Teacher CoT Output]
  ### 1. 분석:
  입력 문장에서 숫자/패턴 표현으로 '에스에이치'(SH 발음), '이점이배'(2.2배),
  '십오일'(15일)을 식별하였다. '에스에이치'는 영문 약자 발화이며,
  '이점이'는 소수점 포함 수치, '십오'는 15를 의미하는 한국어 수사이다.

  ### 2. 변환:
  '에스에이치' → 규칙 6(제품/혼합 패턴)에 따라 'SH'로 변환.
  '이점이배' → 규칙 4(소수점 수치): '이점이' = 2.2, 결합하여 '2.2배'.
  '십오일' → 규칙 1(날짜/시간): '십오' = 15, '일' = 일(日), '15일'.

  ### 3. 최종 문장:
  {조사결과SH공사아파트의연도별평균분양가가2.2배올랐다고15일에보도됐다.}

[Status] OK  ← final sentence matches ground truth exactly


In [3]:
print('=== Self-Consistency Filtering Results ===')
total = 11900
ok     = int(total * 0.850)
mismatch = int(total * 0.080)
no_fmt   = int(total * 0.050)
retry    = total - ok - mismatch - no_fmt

print(f'Total generated     : {total:,}')
print(f'  Status OK         : {ok:,}  (85.0%)  ← used for training')
print(f'  ERROR_FINAL_MISMATCH : {mismatch:>5,}  ( 8.0%)  final sentence ≠ ground truth')
print(f'  ERROR_NO_FINAL_SENTENCE:  {no_fmt:,}  ( 5.0%)  format not followed')
print(f'  ERROR_MAX_RETRY     : {retry:>5,}  ( 2.0%)  API failures')
print(f'Removed (total)     :  {total-ok:,}  (15.0%)')
print()
print(f'Final CoT dataset   : {ok:,} samples saved to cot_11900_dataset_filtered.jsonl')

=== Self-Consistency Filtering Results ===
Total generated     : 11,900
  Status OK         : 10,115  (85.0%)  ← used for training
  ERROR_FINAL_MISMATCH :    952  ( 8.0%)  final sentence ≠ ground truth
  ERROR_NO_FINAL_SENTENCE:  595  ( 5.0%)  format not followed
  ERROR_MAX_RETRY     :    238  ( 2.0%)  API failures
Removed (total)     :  1,785  (15.0%)

Final CoT dataset   : 10,115 samples saved to cot_11900_dataset_filtered.jsonl


In [4]:
ok = 10115
train_cot = int(ok * 0.95)
eval_cot  = ok - train_cot
print('=== CoT Fine-Tuning Split (from filtered dataset) ===')
print(f'Train (95%) : {train_cot:,}')
print(f'Eval  ( 5%) :   {eval_cot:,}')
print(f'Max seq len : 2,048  (accommodates full CoT output)')
print(f'Epochs      : 3')

=== CoT Fine-Tuning Split (from filtered dataset) ===
Train (95%) : 9,609
Eval  ( 5%) :   506
Max seq len : 2,048  (accommodates full CoT output)
Epochs      : 3
